- ALGORITMO: extended_fitness_with_imports
- RISOLUZIONE: 4 digits

In [1]:
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = next(p for p in [Path().resolve(), *Path().resolve().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

from src.algorithms_with_imports import compute_extended_fitness_with_imports
from src.algorithms import compute_alpha

In [2]:
# INPUT: CARTELLE
Mcp_export_FOLDER = Path(r'C:\Users\vitto\Desktop\CSH RESEARCH\data\HS02_all_years\4_digits\alligned\binarized\Mcp_export')
Mcp_import_FOLDER = Path(r'C:\Users\vitto\Desktop\CSH RESEARCH\data\HS02_all_years\4_digits\alligned\binarized\Mcp_import')

# OUTPUT: CARTELLE
countries_FOLDER = Path(r'C:\Users\vitto\Desktop\CSH RESEARCH\results\year_by_year\4_digits\extended_fitness_with_imports\countries')
products_FOLDER  = Path(r'C:\Users\vitto\Desktop\CSH RESEARCH\results\year_by_year\4_digits\extended_fitness_with_imports\products')
countries_FOLDER.mkdir(parents=True, exist_ok=True)
products_FOLDER.mkdir(parents=True, exist_ok=True)

In [3]:
# C e alpha sono fissi: fuori dal ciclo
C = pd.read_parquet(
    r'C:\Users\vitto\Desktop\CSH RESEARCH\data\HS02_all_years\4_digits\alligned\binarized\C\C_4_digits_HS02.parquet')

alpha = compute_alpha(C)

In [7]:
# LEGENDE: anche queste fuori dal ciclo
countries_info = pd.read_csv(
    r"C:\Users\vitto\Desktop\CSH RESEARCH\data\raw\BACI"
    r"\country_codes_V202601.csv",
    usecols=["country_code", "country_name"],
)

products_info = pd.read_csv(
    r"C:\Users\vitto\Desktop\CSH RESEARCH\data\raw\BACI"
    r"\product_codes_HS02_V202601.csv",
    usecols=["code", "description"],
)
products_info["code"] = products_info["code"].astype(str).str.zfill(4)

In [9]:
for export_path in sorted(Mcp_export_FOLDER.glob("*.parquet")):

    match = re.search(r"Y(\d{4})", export_path.name)
    year = int(match.group(1))

    # se l'anno e' gia' stato fatto lo salto (comodo se la pipeline si interrompe)
    if (countries_FOLDER / f"countries_table_Y{year}.parquet").exists():
        print(year, "gia' fatto, salto")
        continue

    import_path = Mcp_import_FOLDER / f"Mcp_import_alligned_Y{year}.parquet"

    Mcp_export = pd.read_parquet(export_path)
    Mcp_import = pd.read_parquet(import_path)

    total_complexity, fitness, input_complexity, intrinsic_complexity, import_penalty = \
        compute_extended_fitness_with_imports(
            Mcp_export, Mcp_import, C.T, alpha=alpha, tensor_window=3, max_iter=1000,
            tol=1e-6, track_convergence=False, save_delta=False, delta_window=None)

    countries_table = pd.DataFrame({
        "year":           year,
        "country":        Mcp_export.index,
        "fitness":        fitness.values,
        "import_penalty": import_penalty.values,
    }).reset_index(drop=True)

    countries_table = countries_table.merge(
        countries_info, left_on="country", right_on="country_code", how="left"
    )

    products_table = pd.DataFrame({
        "year":                 year,
        "product":              Mcp_export.columns.astype(str),
        "total_complexity":     total_complexity.values,
        "input_complexity":     input_complexity.values,
        "intrinsic_complexity": intrinsic_complexity.values,
    }).reset_index(drop=True)

    products_table = products_table.merge(
        products_info, left_on="product", right_on="code", how="left"
    )

    countries_table.to_parquet(countries_FOLDER / f"countries_tableY{year}.parquet")
    products_table.to_parquet(products_FOLDER / f"products_table_Y{year}.parquet")

    print(year, "fatto")

2002 fatto
2003 fatto
2004 fatto
2005 fatto
2006 fatto
2007 fatto
2008 fatto
2009 fatto
2010 fatto
2011 fatto
2012 fatto
2013 fatto
2014 fatto
2015 fatto
2016 fatto
2017 fatto
2018 fatto
2019 fatto
2020 fatto
2021 fatto
2022 fatto
2023 fatto
2024 fatto


- ALGORITMO: extended_eci_with_imports
- RISOLUZIONE: 4 digits

In [10]:
# INPUT: CARTELLE
Mcp_export_FOLDER = Path(r'C:\Users\vitto\Desktop\CSH RESEARCH\data\HS02_all_years\4_digits\alligned\binarized\Mcp_export')
Mcp_import_FOLDER = Path(r'C:\Users\vitto\Desktop\CSH RESEARCH\data\HS02_all_years\4_digits\alligned\binarized\Mcp_import')

# OUTPUT: CARTELLE
countries_FOLDER = Path(r'C:\Users\vitto\Desktop\CSH RESEARCH\results\year_by_year\4_digits\extended_eci_with_imports\countries')
products_FOLDER  = Path(r'C:\Users\vitto\Desktop\CSH RESEARCH\results\year_by_year\4_digits\extended_eci_with_imports\products')
countries_FOLDER.mkdir(parents=True, exist_ok=True)
products_FOLDER.mkdir(parents=True, exist_ok=True)

In [11]:
# C e alpha sono fissi: fuori dal ciclo
C = pd.read_parquet(
    r'C:\Users\vitto\Desktop\CSH RESEARCH\data\HS02_all_years\4_digits\alligned\binarized\C\C_4_digits_HS02.parquet')

alpha = compute_alpha(C)

In [12]:
# LEGENDE: anche queste fuori dal ciclo
countries_info = pd.read_csv(
    r"C:\Users\vitto\Desktop\CSH RESEARCH\data\raw\BACI"
    r"\country_codes_V202601.csv",
    usecols=["country_code", "country_name"],
)

products_info = pd.read_csv(
    r"C:\Users\vitto\Desktop\CSH RESEARCH\data\raw\BACI"
    r"\product_codes_HS02_V202601.csv",
    usecols=["code", "description"],
)
products_info["code"] = products_info["code"].astype(str).str.zfill(4)

In [14]:
for export_path in sorted(Mcp_export_FOLDER.glob("*.parquet")):

    match = re.search(r"Y(\d{4})", export_path.name)
    year = int(match.group(1))

    # se l'anno e' gia' stato fatto lo salto (comodo se la pipeline si interrompe)
    if (countries_FOLDER / f"countries_table_Y{year}.parquet").exists():
        print(year, "gia' fatto, salto")
        continue

    import_path = Mcp_import_FOLDER / f"Mcp_import_alligned_Y{year}.parquet"

    Mcp_export = pd.read_parquet(export_path)
    Mcp_import = pd.read_parquet(import_path)

    total_complexity, fitness, input_complexity, intrinsic_complexity, import_penalty = \
        compute_extended_fitness_with_imports(
            Mcp_export, Mcp_import, C.T, alpha=alpha, tensor_window=3, max_iter=1000,
            tol=1e-6, track_convergence=False, save_delta=False, delta_window=None)

    countries_table = pd.DataFrame({
        "year":           year,
        "country":        Mcp_export.index,
        "fitness":        fitness.values,
        "import_penalty": import_penalty.values,
    }).reset_index(drop=True)

    countries_table = countries_table.merge(
        countries_info, left_on="country", right_on="country_code", how="left"
    )

    products_table = pd.DataFrame({
        "year":                 year,
        "product":              Mcp_export.columns.astype(str),
        "total_complexity":     total_complexity.values,
        "input_complexity":     input_complexity.values,
        "intrinsic_complexity": intrinsic_complexity.values,
    }).reset_index(drop=True)

    products_table = products_table.merge(
        products_info, left_on="product", right_on="code", how="left"
    )

    countries_table.to_parquet(countries_FOLDER / f"countries_table_Y{year}.parquet")
    products_table.to_parquet(products_FOLDER / f"products_table_Y{year}.parquet")

    print(year, "fatto")

2002 fatto
2003 fatto
2004 fatto
2005 fatto
2006 fatto
2007 fatto
2008 fatto
2009 fatto
2010 fatto
2011 fatto
2012 fatto
2013 fatto
2014 fatto
2015 fatto
2016 fatto
2017 fatto
2018 fatto
2019 fatto
2020 fatto
2021 fatto
2022 fatto
2023 fatto
2024 fatto


- ALGORITMO: extended_fitness_with_imports
- RISOLUZIONE: 6 digits

In [ ]:
# INPUT: CARTELLE
Mcp_export_FOLDER = Path(r'C:\Users\vitto\Desktop\CSH RESEARCH\data\HS02_all_years\6_digits\alligned\binarized\Mcp_export')
Mcp_import_FOLDER = Path(r'C:\Users\vitto\Desktop\CSH RESEARCH\data\HS02_all_years\6_digits\alligned\binarized\Mcp_import')

# OUTPUT: CARTELLE
countries_FOLDER = Path(r'C:\Users\vitto\Desktop\CSH RESEARCH\results\year_by_year\6_digits\extended_fitness_with_imports\countries')
products_FOLDER  = Path(r'C:\Users\vitto\Desktop\CSH RESEARCH\results\year_by_year\6_digits\extended_fitness_with_imports\products')
countries_FOLDER.mkdir(parents=True, exist_ok=True)
products_FOLDER.mkdir(parents=True, exist_ok=True)

In [ ]:
# C e alpha sono fissi: fuori dal ciclo
C = pd.read_parquet(
    r'C:\Users\vitto\Desktop\CSH RESEARCH\data\HS02_all_years\6_digits\alligned\binarized\C\C_6_digits_HS02.parquet')

alpha = compute_alpha(C)

In [ ]:
# LEGENDE: anche queste fuori dal ciclo
countries_info = pd.read_csv(
    r"C:\Users\vitto\Desktop\CSH RESEARCH\data\raw\BACI\BACI_HS22"
    r"\LEGENDE\country_codes_V202601.csv",
    usecols=["country_code", "country_name"],
)

products_info = pd.read_csv(
    r"C:\Users\vitto\Desktop\CSH RESEARCH\data\raw\BACI\BACI_HS22"
    r"\LEGENDE\product_codes_HS22_4_V202601.csv",
    usecols=["code", "description"],
)
products_info["code"] = products_info["code"].astype(str).str.zfill(4)

In [ ]:
for export_path in sorted(EXPORT_FOLDER.glob("*.parquet")):

    match = re.search(r"Y(\d{4})", export_path.name)
    year = int(match.group(1))

    # se l'anno e' gia' stato fatto lo salto (comodo se la pipeline si interrompe)
    if (COUNTRIES_FOLDER / f"countries_table_Y{year}.parquet").exists():
        print(year, "gia' fatto, salto")
        continue

    import_path = IMPORT_FOLDER / f"Mcp_import_alligned_Y{year}.parquet"

    Mcp_export = pd.read_parquet(export_path)
    Mcp_import = pd.read_parquet(import_path)

    total_complexity, fitness, input_complexity, intrinsic_complexity, import_penalty = \
        compute_extended_fitness_with_imports(
            Mcp_export, Mcp_import, C.T, alpha=alpha, tensor_window=3, max_iter=1000,
            tol=1e-6, track_convergence=False, save_delta=False, delta_window=None)

    countries_table = pd.DataFrame({
        "year":           year,
        "country":        Mcp_export.index,
        "fitness":        fitness.values,
        "import_penalty": import_penalty.values,
    }).reset_index(drop=True)

    countries_table = countries_table.merge(
        countries_info, left_on="country", right_on="country_code", how="left"
    )

    products_table = pd.DataFrame({
        "year":                 year,
        "product":              Mcp_export.columns.astype(str),
        "total_complexity":     total_complexity.values,
        "input_complexity":     input_complexity.values,
        "intrinsic_complexity": intrinsic_complexity.values,
    }).reset_index(drop=True)

    products_table = products_table.merge(
        products_info, left_on="product", right_on="code", how="left"
    )

    countries_table.to_parquet(COUNTRIES_FOLDER / f"countries_table_Y{year}.parquet")
    products_table.to_parquet(PRODUCTS_FOLDER / f"products_table_Y{year}.parquet")

    print(year, "fatto")




- ALGORITMO: extended_eci_with_imports
- RISOLUZIONE: 6 digits

In [ ]:
# INPUT: CARTELLE
Mcp_export_FOLDER = Path(r'C:\Users\vitto\Desktop\CSH RESEARCH\data\HS02_all_years\6_digits\alligned\binarized\Mcp_export')
Mcp_import_FOLDER = Path(r'C:\Users\vitto\Desktop\CSH RESEARCH\data\HS02_all_years\6_digits\alligned\binarized\Mcp_import')

# OUTPUT: CARTELLE
countries_FOLDER = Path(r'C:\Users\vitto\Desktop\CSH RESEARCH\results\year_by_year\6_digits\extended_eci_with_imports\countries')
products_FOLDER  = Path(r'C:\Users\vitto\Desktop\CSH RESEARCH\results\year_by_year\6_digits\extended_eci_with_imports\products')
countries_FOLDER.mkdir(parents=True, exist_ok=True)
products_FOLDER.mkdir(parents=True, exist_ok=True)

In [ ]:
# C e alpha sono fissi: fuori dal ciclo
C = pd.read_parquet(
    r'C:\Users\vitto\Desktop\CSH RESEARCH\data\HS02_all_years\6_digits\alligned\binarized\C\C_6_digits_HS02.parquet')

alpha = compute_alpha(C)

In [ ]:
# LEGENDE: anche queste fuori dal ciclo
countries_info = pd.read_csv(
    r"C:\Users\vitto\Desktop\CSH RESEARCH\data\raw\BACI\BACI_HS22"
    r"\LEGENDE\country_codes_V202601.csv",
    usecols=["country_code", "country_name"],
)

products_info = pd.read_csv(
    r"C:\Users\vitto\Desktop\CSH RESEARCH\data\raw\BACI\BACI_HS22"
    r"\LEGENDE\product_codes_HS22_4_V202601.csv",
    usecols=["code", "description"],
)
products_info["code"] = products_info["code"].astype(str).str.zfill(4)
for export_path in sorted(EXPORT_FOLDER.glob("*.parquet")):

    match = re.search(r"Y(\d{4})", export_path.name)
    year = int(match.group(1))

    # se l'anno e' gia' stato fatto lo salto (comodo se la pipeline si interrompe)
    if (COUNTRIES_FOLDER / f"countries_table_Y{year}.parquet").exists():
        print(year, "gia' fatto, salto")
        continue

    import_path = IMPORT_FOLDER / f"Mcp_import_alligned_Y{year}.parquet"

    Mcp_export = pd.read_parquet(export_path)
    Mcp_import = pd.read_parquet(import_path)

    total_complexity, fitness, input_complexity, intrinsic_complexity, import_penalty = \
        compute_extended_fitness_with_imports(
            Mcp_export, Mcp_import, C.T, alpha=alpha, tensor_window=3, max_iter=1000,
            tol=1e-6, track_convergence=False, save_delta=False, delta_window=None)

    countries_table = pd.DataFrame({
        "year":           year,
        "country":        Mcp_export.index,
        "fitness":        fitness.values,
        "import_penalty": import_penalty.values,
    }).reset_index(drop=True)

    countries_table = countries_table.merge(
        countries_info, left_on="country", right_on="country_code", how="left"
    )

    products_table = pd.DataFrame({
        "year":                 year,
        "product":              Mcp_export.columns.astype(str),
        "total_complexity":     total_complexity.values,
        "input_complexity":     input_complexity.values,
        "intrinsic_complexity": intrinsic_complexity.values,
    }).reset_index(drop=True)

    products_table = products_table.merge(
        products_info, left_on="product", right_on="code", how="left"
    )

    countries_table.to_parquet(COUNTRIES_FOLDER / f"countries_table_Y{year}.parquet")
    products_table.to_parquet(PRODUCTS_FOLDER / f"products_table_Y{year}.parquet")

    print(year, "fatto")